In [1]:
# Imports

from pathlib import Path
from scipy.io import savemat
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import re

In [2]:
# Define participant and set paths

participant = "vp09_MH"

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError: 
    BASE_DIR = Path.cwd().parent
    
INPUT_PATH = BASE_DIR / "analysis" / "participants" / participant / "logs"
OUTPUT_PATH = BASE_DIR / "analysis" / "participants" / participant

os.chdir(INPUT_PATH)

print(BASE_DIR)
print(INPUT_PATH)
print(OUTPUT_PATH)

D:\thesis-matlab-pipeline
D:\thesis-matlab-pipeline\analysis\participants\vp09_MH\logs
D:\thesis-matlab-pipeline\analysis\participants\vp09_MH


In [3]:
# Get logs in dir

logs = [log for log in os.listdir() if log.endswith(".csv")]

print(logs)

['MH1.csv', 'MH2.csv', 'MH3.csv', 'MH4.csv', 'MH_localizer_logging.csv']


In [4]:
def build_arrays(log):
    log_conditions = set(log["condition"].dropna())
    
    if "coherent" in log_conditions:
        condition_order = ["coherent", "incoherent_real", "incoherent_mock", "neutral", "target", "cue"]
        
    elif "Congruent" in log_conditions:
        condition_order = ["Congruent", "Incongruent_Real", "Incongruent_Fake", "Neutral", "True", "cue"]
        
    elif "Face" in log_conditions:
        condition_order = ["Face", "scene", "scrambled"]
        
    else:
        raise KeyError("Invalid log format!")
    
    columns = len(condition_order)

    onset_array = np.empty((columns,), dtype=np.object_)
    duration_array = np.empty((columns,), dtype=np.object_)
    names_array = np.empty((columns,), dtype=np.object_)

    return condition_order, onset_array, duration_array, names_array

In [ ]:
def write_conditions_and_onsets(log, condition_order, onset_array):
    is_func = "cue" in condition_order
    
    if is_func:
        onset_name = "_onset" if "Congruent" in condition_order else "_onset_cor"
    else:
        onset_name = "time when it onsets"
    
    for i, condition in enumerate(condition_order):
        if is_func:
            if condition == "cue":
                cond_onsets = log[f"s1{onset_name}"].astype(float)
                
            else:
                cond_onsets = log.loc[
                    log["condition"] == condition, f"s2{onset_name}"
                    ].astype(float)
                
        else:
           cond_onsets = log.loc[
                log["condition"] == condition, onset_name
               ].astype(float)
            
           cond_onsets += 4

        onset_array[i] = cond_onsets.to_numpy()[:, None]

    return onset_array

In [6]:
def write_durations(duration_array, condition_order):
    duration_list = [0.20 for c in condition_order]

    for i, duration in enumerate(duration_list):
        duration_array[i] = duration

    return duration_array

In [7]:
def write_names(condition_order, names_array):

    if "cue" in condition_order:
        condition_order = ["Congruent", "Incongruent_Real", "Incongruent_Fake", "Neutral", "Target", "Cue"] 
    
    for i, name in enumerate(condition_order):
        names_array[i] = name

    return names_array

In [8]:
def to_dict(onset_array, durration_array, names_array):
    mat_dict = {
        "onsets": onset_array,
        "names": names_array,
        "durations": duration_array,
    }

    return mat_dict

In [9]:
def prep_dir(log_id: str):
    match = re.search(r"\d+", log_id)
    
    if match:
        log_number = match.group()
        func_dir = OUTPUT_PATH / f"0{log_number}_func"
    else:
        func_dir = OUTPUT_PATH / "00_loc"
    
    func_dir.mkdir(parents=True, exist_ok=True)

    return func_dir

In [10]:
def save_mcf(output_dir: Path, mat_dict: dict):
    out_dir = output_dir / "mcf"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_file = out_dir / "MCF.mat"
    savemat(out_file, mat_dict)

In [11]:
# Run conversion for all files

for log in logs:
    log_path = INPUT_PATH / log
    df = pd.read_csv(log_path)

    condition_order, empty_onset_array, empty_duration_array, empty_name_array = build_arrays(df)
    
    onset_array = write_conditions_and_onsets(df, condition_order, empty_onset_array)
    duration_array = write_durations(empty_duration_array, condition_order)
    names_array = write_names(condition_order, empty_name_array)
    
    mat_dict = to_dict(onset_array, duration_array, names_array)
    
    mcf_dir = prep_dir(log)
    save_mcf(mcf_dir, mat_dict)
    
    print(f"MCF for {log} saved to {mcf_dir}.")
    for array, values in mat_dict.items():
        print(f"\narray")
        print("================")
        for value in values:
            print(value)

print(f"MCF conversion completed for {logs}.")

MCF for MH1.csv saved to D:\thesis-matlab-pipeline\analysis\participants\vp09_MH\01_func.

array
[[ 18.5582983]
 [ 26.5718012]
 [ 42.2657491]
 [ 61.4916982]
 [ 75.752676 ]
 [ 90.9971569]
 [124.2843257]
 [141.7443234]
 [183.31167  ]
 [243.6216064]
 [291.5354736]
 [299.9988272]
 [342.315008 ]
 [376.2514486]
 [449.0221708]
 [482.8419072]
 [516.5950763]]
[[ 10.7780191]
 [ 99.6104262]
 [116.5205821]
 [150.2576323]
 [174.7484017]
 [191.5087055]
 [218.5149774]
 [332.5189789]
 [360.8575338]
 [386.4972206]
 [432.2622115]
 [441.3418675]
 [524.9918446]]
[[ 52.4951605]
 [106.7743164]
 [166.3183022]
 [210.5845993]
 [225.528781 ]
 [252.7845122]
 [308.2620391]
 [368.5877617]
 [394.5107141]
 [422.4995112]
 [473.7622208]
 [491.121924 ]
 [507.7487099]]
[[ 68.5056273]
 [ 82.7667697]
 [132.3478046]
 [157.7881741]
 [273.4761773]
 [282.8558654]
 [317.0917632]
 [324.9888331]
 [352.5275451]
 [404.2567707]
 [413.5195891]
 [466.8150184]
 [499.0022111]]
[[ 34.2687062]
 [200.4051126]
 [234.75849  ]
 [459.0181001]